In [ ]:
# GORI SAMPLE LAB — Motor gratuito de criação de matéria-prima musical
# Execute esta única célula no Google Colab. Usa Stable Audio 3 Small Music.
# Cole abaixo o campo "Brief musical" gerado pelo GORI SAMPLE LAB.

import os, sys, subprocess, textwrap

BRIEF = """
Componha uma progressão instrumental original no estilo soul/funk brasileiro dos anos 70 em D menor, a 88 BPM. Utilize um piano Rhodes com timbre quente e empilhado com chiado de vinil, tocando acordes melancólicos e abertos. Adicione um baixo elétrico com timbres arredondados fazendo linhas rítmicas e espaçadas. O arranjo deve ser orgânico, espaçado e contemplativo, com variações sutis nas frases do Rhodes e sem percussão.
"""

DURATION_SECONDS = 60
OUTPUT_FILE = "/content/GORI_SAMPLE_LAB_SAMPLE.wav"

print("1/4 — Corrigindo PyTorch e Torchaudio...")
# O Stable Audio 3 fixa os dois pacotes na mesma versão.
# Instalar o par CUDA 12.6 evita incompatibilidade de bibliotecas binárias.
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q", "--upgrade", "--force-reinstall",
    "torch==2.7.1", "torchaudio==2.7.1",
    "--index-url", "https://download.pytorch.org/whl/cu126"
])

print("2/4 — Instalando o motor Stable Audio 3...")
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "git+https://github.com/Stability-AI/stable-audio-3.git"
])

print("3/4 — Preparando composição...")
import torch
import torchaudio
from stable_audio_3 import StableAudioModel
print("PyTorch:", torch.__version__, "| Torchaudio:", torchaudio.__version__)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Dispositivo:", device)
if device == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

# O briefing da análise é tratado como referência estética.
# O modelo recebe instruções adicionais para criar uma peça-fonte,
# e não um beat, loop de bateria ou instrumental de rap.
prompt = f"""
Create a completely original, fully instrumental source-music recording for a hip-hop beatmaker to sample, chop, pitch and rearrange.
This must sound like a real, richly arranged vintage studio recording, not a hip-hop beat, not a backing track and not a short repetitive loop.
Musical reference brief: {BRIEF}

Develop a complete 60-second musical passage with a clear beginning, evolving middle and intentional ending.
Build several independent musical layers with distinct roles: expressive chordal instrument, a melodic bass part, one or two complementary melodic instruments selected to fit the reference, and subtle background harmonic textures.
Use memorable original melodic phrases, answering phrases, counter-melodies, harmonic movement, chord inversions, extended voicings, passing chords, fills and small variations.
Create contrasting sections: an intimate opening motif, a fuller ensemble passage, a contrasting middle section with a different melodic or harmonic color, then a transformed return and natural ending.
Make instruments interact like musicians playing together in a live studio: expressive phrasing, human timing, natural dynamics, small performance imperfections, room ambience, tape warmth and gentle analog saturation.
Keep the source music spacious enough to be chopped into usable sections, but musically rich and harmonically interesting throughout.
Strictly no drums, no percussion, no kick, no snare, no hi-hats, no cymbals, no claps, no snaps, no beatboxing, no rhythm loop, no rap, no vocals, no spoken words, no DJ scratches.
Do not imitate any existing artist, song, sample, melody, recording or recognizable arrangement.
"""

negative_prompt = (
    "drum kit, drums, percussion, kick drum, snare drum, hi hats, cymbals, "
    "claps, snaps, trap beat, hip hop beat, boom bap beat, electronic beat, "
    "rap, vocals, singing, spoken word, DJ scratches, repetitive 2 bar loop, "
    "simple one chord loop, thin arrangement, harsh digital sound, low quality"
)

print("Carregando Stable Audio 3 Small Music (primeira execução pode demorar)...")
model = StableAudioModel.from_pretrained("small-music", device=device)

print("4/4 — Gerando matéria-prima musical. Aguarde...")
with torch.inference_mode():
    audio = model.generate(
        prompt=prompt,
        negative_prompt=negative_prompt,
        duration=DURATION_SECONDS,
        steps=8,
        cfg_scale=1.0,
        seed=-1,
        batch_size=1
    )

audio = audio.detach().float().cpu()
if audio.ndim == 3:
    audio_to_save = audio[0]
elif audio.ndim == 2:
    audio_to_save = audio
else:
    raise RuntimeError(f"Formato de áudio inesperado: {tuple(audio.shape)}")

torchaudio.save(OUTPUT_FILE, audio_to_save, model.model.sample_rate)
print("\nSample criado:", OUTPUT_FILE)
print("Duração aproximada:", round(audio_to_save.shape[-1] / model.model.sample_rate, 1), "segundos")
print("Formato: WAV estéreo, 44.1 kHz (conforme configuração do modelo)")

from IPython.display import Audio, display
display(Audio(OUTPUT_FILE))
from google.colab import files
files.download(OUTPUT_FILE)
